# Covariate-scaled multinomial logit

Estimate utility scales as functions of an observation-level covariate.

This notebook is self-contained and was executed in the repository's Office
validation environment. Install the released package with
`pip install torchdcm`, then select CPU or CUDA through the `device` argument.


In [1]:
import numpy as np
import pandas as pd
import torch

from torchdcm import (
    AlternativeScale,
    Beta,
    ChoiceDataset,
    ChoiceLatentEffect,
    ContinuousIndicator,
    CovariateScale,
    CovariateScaledMultinomialLogit,
    CrossNest,
    CrossNestedLogit,
    ErrorComponent,
    ErrorComponentsLogit,
    HybridChoiceModel,
    LatentClassLogit,
    LatentVariable,
    MixedLogit,
    MultinomialLogit,
    Nest,
    NestedLogit,
    OrderedChoiceDataset,
    OrderedLogit,
    OrderedProbit,
    RandomCoefficient,
    ScaledMultinomialLogit,
    UtilitySpec,
    WTPCoefficient,
    WTPMixedLogit,
)
from torchdcm.datasets import make_swissmetro_like

torch.set_default_dtype(torch.float64)
torch.manual_seed(7)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"TorchDCM example running on {device}")


def show_result(result):
    print(f"Final log likelihood: {result.loglike:.6f}")
    print(f"AIC: {result.aic:.3f}; BIC: {result.bic:.3f}")
    return pd.DataFrame(
        {
            "estimate": result.values,
            "std. error": result.bse,
            "z": result.zvalues,
            "p-value": result.pvalues,
        },
        index=pd.Index(result.param_names, name="parameter"),
    ).round(6)


TorchDCM example running on cuda


In [2]:
def make_choice_data(n_obs=180, seed=7, *, observation_variables=None):
    frame = make_swissmetro_like(n_obs=n_obs, seed=seed)
    data = ChoiceDataset.from_wide(
        frame,
        alternatives=["TRAIN", "SM", "CAR"],
        choice="choice",
        variables={
            "time": {
                "TRAIN": "time_train",
                "SM": "time_sm",
                "CAR": "time_car",
            },
            "cost": {
                "TRAIN": "cost_train",
                "SM": "cost_sm",
                "CAR": "cost_car",
            },
        },
        obs_variables=observation_variables,
        availability={
            "TRAIN": "avail_train",
            "SM": "avail_sm",
            "CAR": "avail_car",
        },
        individual_id="person_id",
    )
    return frame, data


def make_utility_spec(suffix=""):
    tag = f"_{suffix}" if suffix else ""
    spec = UtilitySpec()
    spec.utility(
        "TRAIN",
        Beta(f"ASC_TRAIN{tag}", init=0.0)
        + Beta(f"B_TIME{tag}", init=-0.01) * "time"
        + Beta(f"B_COST{tag}", init=-0.10) * "cost",
    )
    spec.utility(
        "SM",
        Beta(f"B_TIME{tag}", init=-0.01) * "time"
        + Beta(f"B_COST{tag}", init=-0.10) * "cost",
    )
    spec.utility(
        "CAR",
        Beta(f"ASC_CAR{tag}", init=0.0)
        + Beta(f"B_TIME{tag}", init=-0.01) * "time"
        + Beta(f"B_COST{tag}", init=-0.10) * "cost",
    )
    return spec

frame = make_swissmetro_like(n_obs=180, seed=17)
frame["income_std"] = (
    (frame["income"] - frame["income"].mean()) / frame["income"].std()
)
data = ChoiceDataset.from_wide(
    frame,
    alternatives=["TRAIN", "SM", "CAR"],
    choice="choice",
    variables={
        "time": {"TRAIN": "time_train", "SM": "time_sm", "CAR": "time_car"},
        "cost": {"TRAIN": "cost_train", "SM": "cost_sm", "CAR": "cost_car"},
        "income_std": {
            "TRAIN": "income_std",
            "SM": "income_std",
            "CAR": "income_std",
        },
    },
    availability={
        "TRAIN": "avail_train",
        "SM": "avail_sm",
        "CAR": "avail_car",
    },
    individual_id="person_id",
)
spec = make_utility_spec()


In [3]:
scales = {
    "TRAIN": CovariateScale(
        log_scale=Beta("GAMMA_TRAIN", init=0.05) * "income_std"
    ),
    "SM": CovariateScale(value=1.0),
    "CAR": CovariateScale(
        log_scale=Beta("GAMMA_CAR", init=-0.05) * "income_std"
    ),
}
model = CovariateScaledMultinomialLogit(
    spec,
    scales,
    device=device,
    max_iter=60,
)
result = model.fit(data)
show_result(result)


Final log likelihood: -170.213165
AIC: 352.426; BIC: 371.584


,estimate,std. error,z,p-value
parameter,,,,
ASC_TRAIN,0.648108,0.353500,1.833405,0.066742
B_TIME,-0.041092,0.009449,-4.348975,0.000014
B_COST,-0.075692,0.039856,-1.899162,0.057543
ASC_CAR,0.864704,0.315551,2.740302,0.006138
GAMMA_TRAIN,-0.010930,0.039721,-0.275172,0.783184
GAMMA_CAR,-0.003958,0.046927,-0.084343,0.932784
